# BL00 Epoch-100 independent G4 + terminal audit
**Read-only · no training · no checkpoint unpickling · B-TEST sealed.**
Audit the authenticated saved-version output at `/kaggle/input/notebooks/hebut123/grazpedwri-dx-bl00-ddp-b16/BL00_CONTINUOUS_B16_OUTPUT`. If the session output is still local, the cell can also read `/kaggle/working/BL00_CONTINUOUS_B16_OUTPUT`. Prefer the saved-version mount when available. The script checks exact SHA-256 of both G4 rank checkpoints, selected-best and last weights, policy provenance, training receipt, and actual per-epoch G4 worker events. The selected-best B-VAL result **must still be evaluated independently**; last epoch metrics are not best-checkpoint metrics.


In [ ]:
# BL00 Epoch-100 G4 / terminal / policy audit. READ ONLY. No GPU, no training, no B-TEST.
from pathlib import Path
import hashlib, json, os

ROOTS = [
    Path("/kaggle/input/notebooks/hebut123/grazpedwri-dx-bl00-ddp-b16/BL00_CONTINUOUS_B16_OUTPUT"),
    Path("/kaggle/working/BL00_CONTINUOUS_B16_OUTPUT"),
]
if os.environ.get("BL00_AUDIT_ROOT"):
    ROOTS.insert(0, Path(os.environ["BL00_AUDIT_ROOT"]))
M100="ca44131b59ee8d2ba393f01c0cc642c1da26e0f0f1966a637d740d62922a67ae"
M095="05a87470d32ec7506f9d4212ce7ba5881ec703299883c2c445505bf2bc6e0c5d"
POLICY="62fc6d96f84b7348f2ab481b3d894fc040be1613c7417dc8e775ac902a8d928c"
SOURCE="f440d329d4544ec08ad63c1972bcf9a8b8a7eed8"

def need(condition, name):
    if not condition:
        raise RuntimeError("BL00_AUDIT_FAIL_"+name)
    print("AUDIT PASS:",name,flush=True)

def sha(path):
    need(path.is_file() and not path.is_symlink(), "REGULAR_FILE_"+path.name)
    h=hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda:f.read(1<<20),b""):
            h.update(chunk)
    return h.hexdigest()

def obj(path):
    need(path.is_file() and not path.is_symlink(),"JSON_EXISTS_"+path.name)
    return json.loads(path.read_text("utf-8"))

roots=[p for p in ROOTS if p.is_dir()]
need(len(roots)>0, "SAVED_OUTPUT_AVAILABLE")
# If saved version still references an older session, we must not silently accept it.
root=roots[0]
print("AUDIT SOURCE:",root)
g4=root/"g4"
t=root/"terminal"
manifest_path=g4/"BL00_EPOCH100_two_rank_manifest.json"
need(sha(manifest_path)==M100, "PINNED_EPOCH100_MANIFEST_SHA")
m=obj(manifest_path)
ptr=obj(g4/"LATEST_MANIFEST.json")
need(ptr.get("manifest")==manifest_path.name and
     ptr.get("manifest_sha256")==M100 and
     ptr.get("completed_epochs")==100,"LATEST_POINTER")
need(m.get("schema")=="BL00_CONT_COMPLETE_G4_MANIFEST_V1" and
     m.get("complete") is True and m.get("arm")=="BL00" and
     m.get("world_size")==2 and m.get("completed_epochs")==100 and
     m.get("next_epoch_index")==100 and m.get("optimizer_updates")==22880 and
     m.get("frozen_commit")==SOURCE and m.get("train_images")==14227 and
     m.get("operational_val_images")==3049 and m.get("steps_per_rank")==890 and
     m.get("b_test_access")=="NONE","G4_METADATA")
need(m.get("amp128_recovery_branch") is True and
     m.get("amp128_policy_sha256")==POLICY and
     m.get("amp_scale_at_checkpoint")==128.0 and
     m.get("scientific_original_trajectory") is False,"G4_NUMERICAL_POLICY")
files=m.get("files",{})
need(set(files)=={"BL00_EPOCH100_rank0.pt","BL00_EPOCH100_rank1.pt",
                  "BL00_EPOCH100_best.pt"},"G4_MEMBER_SET")
for name,digest in files.items():
    need(Path(name).name==name and len(digest)==64 and
         sha(g4/name)==digest,"G4_MEMBER_SHA_"+name)
r=m.get("ranks",{})
need(set(r)=={"0","1"} and all(
     r[str(i)].get("rank")==i and
     r[str(i)].get("filename")==f"BL00_EPOCH100_rank{i}.pt" and
     r[str(i)].get("sha256")==files[f"BL00_EPOCH100_rank{i}.pt"] and
     r[str(i)].get("optimizer_updates")==22880 and
     r[str(i)].get("ema_updates")==22880 and
     r[str(i)].get("sampler_next_epoch")==100 for i in (0,1)),
     "TWO_RANK_METADATA")
need(m.get("best_filename")=="BL00_EPOCH100_best.pt" and
     m.get("selected_best_sha256")==files["BL00_EPOCH100_best.pt"],
     "G4_BEST_IDENTITY")

p=obj(root/"AMP128_RECOVERY_POLICY.json")
need(sha(root/"AMP128_RECOVERY_POLICY.json")==POLICY and
     sha(g4/"AMP128_RECOVERY_POLICY.json")==POLICY and
     p.get("arm")=="BL00" and
     p.get("starting_manifest_sha256")==M095 and
     p.get("starting_optimizer_updates")==21767 and
     p.get("original_restored_amp_scale")==1024 and
     p.get("amp_scale")==128 and
     p.get("scientific_original_trajectory") is False and
     p.get("b_test_access")=="NONE", "AMP128_PROVENANCE_POLICY")

terminal=obj(t/"TERMINAL.json")
need(terminal.get("schema")=="BL00_DDP_B16_TERMINAL_AMP128_V1" and
     terminal.get("completed_epochs")==100 and
     terminal.get("optimizer_updates")==22880 and
     terminal.get("operational_val_images")==3049 and
     terminal.get("latest_g4_manifest_sha256")==M100 and
     terminal.get("amp128_policy_sha256")==POLICY and
     terminal.get("scientific_original_trajectory") is False and
     terminal.get("b_test_access")=="NONE","TERMINAL_IDENTITY")
tf=terminal.get("files_sha256",{})
need(set(tf)=={"best.pt","last.pt","AMP128_RECOVERY_POLICY.json"},
     "TERMINAL_MEMBER_SET")
for name,digest in tf.items():
    need(Path(name).name==name and sha(t/name)==digest,
         "TERMINAL_SHA_"+name)
need(tf["best.pt"]==m["selected_best_sha256"] and
     tf["last.pt"]==m["native_last_sha256"] and
     tf["AMP128_RECOVERY_POLICY.json"]==POLICY,
     "TERMINAL_G4_EXACT_WEIGHT_IDENTITY")

result=obj(root/"BL00_CONTINUATION_RESULT.json")
need(result.get("status")=="PASS_RESUME_TRAIN_BVAL_G4" and
     result.get("starting_completed_epochs")==95 and
     result.get("completed_epochs")==100 and
     result.get("optimizer_updates")==22880 and
     result.get("manifest_sha256")==M100 and
     result.get("source_manifest_sha256")==M095 and
     result.get("amp128_policy_sha256")==POLICY and
     result.get("b_test_access")=="NONE","SESSION_RESULT_IDENTITY")
val=m.get("val_metrics",{})
need(terminal.get("metrics")==val and
     abs(float(val["metrics/mAP50-95(B)"])-0.40574)<0.00001 and
     abs(float(result["val_map5095"])-float(val["metrics/mAP50-95(B)"]))<1e-12,
     "LAST_EPOCH_BVAL_CONSISTENCY")

# The actual worker's branch and epoch G4 events are recorded in stdout.
logfile=root/"WORKER_STDOUT.log"
need(logfile.is_file(),"WORKER_STDOUT_EXISTS")
markers=[
"BL00_AMP128_GOVERNED_BRANCH=",
"BL00_AMP128_PASSED_PREVIOUS_FAILURE_BATCH",
*["BL00_CONT_EPOCH_G4_PASS "+str(i) for i in range(96,101)]
]
found={x:0 for x in markers}
branch_info=None
with logfile.open("r",encoding="utf-8",errors="replace") as f:
    for line in f:
        if line.startswith("BL00_AMP128_GOVERNED_BRANCH="):
            branch_info=json.loads(line.split("=",1)[1])
        for marker in markers:
            if marker in line:
                found[marker]+=1
for marker,count in found.items():
    need(count>=1,"WORKER_"+marker.split(" ")[0].replace("=",""))
need(found["BL00_AMP128_GOVERNED_BRANCH="]==1 and
     isinstance(branch_info,dict) and
     branch_info.get("fixed_scale")==128 and
     branch_info.get("original_scale")==1024 and
     branch_info.get("policy_sha256")==POLICY,
     "AMP128_BRANCH_SINGLE_AND_PINNED")

report={
"schema":"BL00_EPOCH100_INDEPENDENT_AUDIT_V1",
"status":"PASS_G4_TERMINAL_PROVENANCE_LAST_EPOCH_BVAL",
"root":str(root),
"completed_epochs":100,
"optimizer_updates":22880,
"epoch100_manifest_sha256":M100,
"terminal_best_sha256":tf["best.pt"],
"terminal_last_sha256":tf["last.pt"],
"policy_sha256":POLICY,
"epoch100_last_bval":val,
"amp128_branch":branch_info,
"selected_best_checkpoint_independent_BVAL":"PENDING",
"b_test_access":"NONE"}
print("="*72)
print("BL00 EPOCH100 INDEPENDENT READ-ONLY AUDIT: PASS")
print("EPOCH100 LAST B-VAL mAP50-95:",val["metrics/mAP50-95(B)"])
print("TERMINAL SELECTED BEST SHA256:",tf["best.pt"])
print("NEXT: independent evaluation of selected best checkpoint on B-VAL only")
print("B-TEST: SEALED")
print("="*72)
print(json.dumps(report,indent=2,sort_keys=True))
